# A QMCPy Quick Start

This notebook mirrors the structure of QMCPy's `quickstart.ipynb` while using the Julia `QMC.jl` APIs.

Original QMCPy demo: [`QMCPy/demos/quickstart.ipynb`](../../QMCPy/demos/quickstart.ipynb)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/QMCSoftware/QMC.jl/blob/develop/demos/quickstart.ipynb)


Consider the problem of integrating the Keister function with respect to a $d$-dimensional Gaussian measure:

$$f(\boldsymbol{x}) = \pi^{d/2} \cos(\lVert\boldsymbol{x}\rVert), \qquad \boldsymbol{x} \in \mathbb{R}^d, \qquad \boldsymbol{X} \sim \mathcal{N}(\boldsymbol{0}_d, \mathsf{I}_d/2),$$

$$\mu = \mathbb{E}[f(\boldsymbol{X})] := \int_{\mathbb{R}^d} f(\boldsymbol{x}) \, \pi^{-d/2} \exp(-\lVert\boldsymbol{x}\rVert^2) \, \mathrm{d}\boldsymbol{x}. $$

This is one of the standard benchmark problems used throughout QMCPy and QMC.jl because the Gaussian true measure, the oscillatory integrand, and the exact value are all available in a compact form.


In [1]:
using QMC
using Printf

function keister(x)
    d = size(x, 2)
    norm_x = sqrt.(sum(x .^ 2, dims=2))
    return vec(π^(d / 2) .* cos.(norm_x))
end


keister (generic function with 1 method)

In addition to the Keister integrand and Gaussian true measure, we must select a discrete distribution and a stopping criterion. The stopping criterion determines how many samples are needed to meet the requested absolute tolerance, while the discrete distribution determines where the integrand is evaluated.

For this example, we follow the QMCPy quickstart and use a lattice rule together with the guaranteed lattice cubature stopping criterion.


In [2]:
d = 2
discrete_distrib = Lattice(d)
true_measure = Gaussian(discrete_distrib; mean=0.0, covariance=1 / 2)
integrand = CustomFun(true_measure, keister)
stopping_criterion = CubQMCLatticeG(integrand; abs_tol=1e-3)


CubQMCLatticeG(abs_tol=0.001, fft)

Calling `integrate` on the stopping criterion returns a `QMCResult` with the numerical solution and a dictionary of diagnostic data. The exact Keister value is also available in `QMC.jl`, so we print both for comparison.


In [3]:
result = integrate(stopping_criterion)
println(result)
@printf("solution        %.6f\n", result.solution)
@printf("exact           %.6f\n", keister_exact(d))
@printf("error           %.6f\n", abs(result.solution - keister_exact(d)))
n_total = get(result.data, :n_total, get(result.data, :n, missing))
println("n_total         ", n_total)

@assert isfinite(result.solution)
@assert abs(result.solution - keister_exact(d)) <= 0.005
@assert n_total > 0


QMCResult(solution=1.808166e+00, n_total=8192, error_bound=5.09e-04)


solution        1.808166
exact           1.808186
error           0.000021
n_total         

8192


## References

This guide is only a quick introduction to the `QMC.jl` workflow. The other Julia demos expand on discrete distributions, stopping criteria, option pricing, and specialized true measures in more detail.
